# 3 - External validation

Tests the signature (Table 2, from notebook 2) on GSE267650, a mouse series that took no
part in its construction (Section 4.6): non-operated controls, ischemia without
reperfusion, and animals reperfused for 4-72 h. Two comparisons are scored: reperfused vs
non-operated controls (`vs_controls`) and reperfused vs ischemia-only (`vs_ischemia`).

All numbers come from one limma fit per comparison, on log2 counts per million:
concordance, median oriented log2 fold change, and the permutation test of the mean
oriented moderated t (1,000 random reassignments of the group labels).

| Output | Use |
|---|---|
| `results/validation_summary.csv` | the numbers of Section 2.4 |
| `results/validation_genes_vs_controls.csv`, `results/validation_genes_vs_ischemia.csv` | one row per signature gene |
| `results/validation_permutations_vs_controls.csv`, `results/validation_permutations_vs_ischemia.csv` | the permutation statistics (row `perm = 0` holds the observed values) |
| `results/validation_design.csv` | groups and numbers of animals |
| `results/TableS6_concordance_by_direction.csv` | Table S6: concordance by direction of the signature genes |

In [1]:
# Move to the project root (the folder that contains run_all.ps1), so that relative
# paths work whether the notebook is started from notebook/ or from the root.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "run_all.ps1")) && dirname(root) != root) {
  root <- dirname(root)
}
if (!file.exists(file.path(root, "run_all.ps1"))) {
  stop("project root not found: no run_all.ps1 above ", getwd())
}
setwd(root)
.libPaths(c(file.path(root, "env", "rlib"), .libPaths()))
Sys.setenv(RENAL_IRI_ROOT = root)
options(stringsAsFactors = FALSE, warn = 1)
cat(R.version.string, "\nroot: ", root, "\n", sep = "")

R version 4.4.3 (2025-02-28)
root: /Users/robertocasale/Desktop/lavoro_FBF/renal_shield_perfusion


## Validation

`R/validation_GSE267650.R`; the permutation test takes a few minutes.

In [2]:
source("R/validation_GSE267650.R")

[2026-10-08 14:43:33] INFO  WROTE       results/validation_design.csv (4 rows)



[2026-10-08 14:43:42] INFO    Mus musculus: 451 outdated rodent symbols updated before ortholog mapping



[2026-10-08 14:43:53] INFO  WROTE       results/validation_genes_vs_controls.csv (256 rows)



[2026-10-08 14:46:08] INFO    Mus musculus: 450 outdated rodent symbols updated before ortholog mapping



[2026-10-08 14:46:10] INFO  WROTE       results/validation_genes_vs_ischemia.csv (256 rows)



[2026-10-08 14:48:24] INFO  WROTE       results/validation_summary.csv (2 rows)



  comparison n_reference n_reperfused signature_genes_measured concordant
 vs_controls           5           31                      256        249
 vs_ischemia           5           31                      256        241
 concordance median_oriented_logFC mean_t permutations r        p
      0.9727                 1.573  6.600         1000 0 0.000999
      0.9414                 1.293  5.669         1000 0 0.000999


## Concordance by direction (Table S6)

Concordance counted separately for the 237 signature genes that go up and the 19 that go
down, because the total is dominated by the upregulated genes. The cell also checks that
every signature gene has a finite, non-zero log2 fold change in both comparisons.

In [3]:
# Table S6: concordance in GSE267650 counted separately for the signature genes that go up
# and for those that go down, in each comparison. It reads the gene-level tables written
# above and checks that every signature gene has a finite, non-zero log2 fold change.
sig  <- read.csv("results/Table2_signature.csv")
sig  <- sig[sig$signature, c("human", "direction")]
summ <- read.csv("results/validation_summary.csv")
s6 <- do.call(rbind, lapply(c("vs_controls", "vs_ischemia"), function(cmp) {
  g <- read.csv(sprintf("results/validation_genes_%s.csv", cmp))
  stopifnot(nrow(g) == nrow(sig), !anyDuplicated(g$human), setequal(g$human, sig$human),
            all(g$direction %in% c("up", "down")),
            identical(g$direction, sig$direction[match(g$human, sig$human)]),
            all(is.finite(g$logFC)), all(g$logFC != 0),
            identical(g$concordant, sign(g$logFC) == ifelse(g$direction == "up", 1, -1)),
            identical(sum(g$concordant), summ$concordant[summ$comparison == cmp]))
  do.call(rbind, lapply(c("up", "down", "all"), function(d) {
    k <- if (d == "all") rep(TRUE, nrow(g)) else g$direction == d
    data.frame(comparison = cmp, direction = d, genes = sum(k),
               concordant = sum(g$concordant[k]), concordance = mean(g$concordant[k]))
  }))
}))
save_csv(s6, "TableS6_concordance_by_direction.csv")
cat("All", nrow(sig), "signature genes have a finite, non-zero log2 fold change in both comparisons.\n")
print(s6, row.names = FALSE, digits = 4)

[2026-10-08 14:48:25] INFO  WROTE       results/TableS6_concordance_by_direction.csv (6 rows)



All 256 signature genes have a finite, non-zero log2 fold change in both comparisons.


  comparison direction genes concordant concordance
 vs_controls        up   237        231      0.9747
 vs_controls      down    19         18      0.9474
 vs_controls       all   256        249      0.9727
 vs_ischemia        up   237        223      0.9409
 vs_ischemia      down    19         18      0.9474
 vs_ischemia       all   256        241      0.9414
